# SpectraLift example integration into HyperBench

This notebook brings the full HyperBench workflow together in one place on the SpectraLift method.

The goal of this notebook is to show how a user could integrate their own real world research models into the HyperBench framework and simply start generating experiments.

## Imports

In [1]:
# HyperBench imports
from pathlib import Path
import csv

import cv2
import numpy as np

from hyperbench import (
    __version__,
    BenchmarkConfig,
    DegradationSpec,
    PipelineAdapter,
    load_hsi,
    normalize_image,
    print_data_stats,
    run_benchmark,
)

print("HyperBench version:", __version__)

# SpectraLift imports
import scipy.io as sio
import numpy as np
from tqdm import tqdm
import os
import math
import time
import io as iot
import sys
import tensorflow as tf
from tensorflow.keras.layers import Dense, ReLU, LeakyReLU
from tensorflow.keras.models import Model
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2

HyperBench version: 0.1.0


2026-03-31 12:10:20.169631: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-31 12:10:22.490159: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-31 12:10:23.306875: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-31 12:10:23.550881: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-31 12:10:25.120780: I tensorflow/core/platform/cpu_feature_guar

## Scene configuration

This notebook assumes a MATLAB scene is available locally.

In [2]:
SCENE_PATH = Path("../../data/DC_data.mat")
SCENE_KEY = "dc"

OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Scene path:", SCENE_PATH)
print("Scene key:", SCENE_KEY)

Scene path: ../../data/DC_data.mat
Scene key: dc


## The SpectraLift implementation code (taken from the author GitHub and lightly adapted to pass metrics to HyperBench instead of printing them)

In [3]:
def numpy_to_tf(np_array):
    """Convert a NumPy array to a TensorFlow tensor."""
    return tf.constant(np_array, dtype=tf.float32)


def tf_to_numpy(tf_tensor):
    """Convert a TensorFlow tensor to a NumPy array."""
    return tf_tensor.numpy()


def apply_srf_tf(hsi, srf):
    """
    Apply an SRF to an HSI in TensorFlow.

    Parameters
    ----------
    hsi : tf.Tensor
        Hyperspectral image of shape (h, w, C)
    srf : tf.Tensor
        Spectral response function of shape (c, C)

    Returns
    -------
    tf.Tensor
        Low-resolution MSI of shape (h, w, c)
    """
    srf_t = tf.transpose(srf)  # (C, c)
    msi = tf.tensordot(hsi, srf_t, axes=[[-1], [0]])
    return msi


def prepare_inputs(hr_msi, lr_hsi, srf):
    """
    Prepare all inputs required by SpectraLift.

    HyperBench passes NumPy arrays to run_pipeline(...). SpectraLift converts
    them to TensorFlow here and constructs the low-resolution MSI used for
    training the spectral MLP.
    """
    hr_msi = numpy_to_tf(hr_msi)
    lr_hsi = numpy_to_tf(lr_hsi)
    srf = numpy_to_tf(srf)
    lr_msi = apply_srf_tf(lr_hsi, srf)
    return hr_msi, lr_hsi, lr_msi, srf


def get_gpu_memory_mb():
    """
    Return current GPU memory usage in MB for GPU:0.

    If no compatible GPU is available, return NaN rather than failing.
    """
    try:
        mem_info = tf.config.experimental.get_memory_info("GPU:0")
        return mem_info["current"] / (1024 ** 2)
    except Exception:
        return float("nan")


def infer_and_analyze_model_performance_tf(model, sample_inputs):
    """
    Analyze model complexity and inference behavior.

    Returns
    -------
    SR_image : tf.Tensor
        Model prediction
    num_params : int
        Number of trainable parameters
    flops : int or float
        Estimated FLOPs
    mem_used : float
        Estimated GPU memory consumption in MB
    inference_time : float
        Inference time in seconds
    """
    input_signature = [
        tf.TensorSpec(shape=inp.shape, dtype=inp.dtype) for inp in sample_inputs
    ]

    @tf.function
    def model_fn(msi):
        return model(msi)

    concrete_func = model_fn.get_concrete_function(*input_signature)
    frozen_func = convert_variables_to_constants_v2(concrete_func)
    graph_def = frozen_func.graph.as_graph_def()

    try:
        original_stdout = sys.stdout
        sys.stdout = iot.StringIO()

        with tf.Graph().as_default() as graph:
            tf.compat.v1.import_graph_def(graph_def, name="")
            run_meta = tf.compat.v1.RunMetadata()
            opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
            opts["output"] = "none"
            prof = tf.compat.v1.profiler.profile(
                graph=graph,
                run_meta=run_meta,
                options=opts,
            )
            flops = prof.total_float_ops if prof is not None else float("nan")
    except Exception:
        flops = float("nan")
    finally:
        sys.stdout = original_stdout

    num_params = int(np.sum([np.prod(v.shape) for v in model.trainable_variables]))
    start_mem = get_gpu_memory_mb()

    start = time.perf_counter()
    SR_image = model(*sample_inputs)
    end = time.perf_counter()
    inference_time = end - start

    end_mem = get_gpu_memory_mb()
    mem_used = end_mem - start_mem if not np.isnan(end_mem) and not np.isnan(start_mem) else float("nan")

    return SR_image, num_params, flops, mem_used, inference_time


def batched_inference(model, hr_msi, batch_size):
    """Run tiled inference over HR_MSI."""
    if isinstance(hr_msi, tf.Tensor):
        hr_msi = hr_msi.numpy()

    H, W, C = hr_msi.shape
    bs = batch_size or max(H, W)
    ys = list(range(0, H, bs))
    xs = list(range(0, W, bs))
    total_batches = len(ys) * len(xs)

    SR = np.zeros((H, W, model.num_outputs), np.float32)

    pbar = tqdm(total=total_batches, desc="Inference")
    for y0 in ys:
        for x0 in xs:
            patch = hr_msi[y0:y0 + bs, x0:x0 + bs]
            patch_tf = tf.constant(patch, tf.float32)
            sr_patch = model(patch_tf).numpy()
            SR[y0:y0 + bs, x0:x0 + bs] = sr_patch
            pbar.update(1)
    pbar.close()

    return SR


class SpectralSR_MLP(Model):
    """SpectraLift spectral MLP."""

    def __init__(self, num_outputs, hidden_size=128):
        super().__init__()
        self.num_outputs = num_outputs

        self.layer1 = Dense(hidden_size, activation=LeakyReLU(alpha=0.3), dtype=tf.float32)
        self.layer2 = Dense(hidden_size, activation=LeakyReLU(alpha=0.3), dtype=tf.float32)
        self.layer3 = Dense(hidden_size, activation=LeakyReLU(alpha=0.3), dtype=tf.float32)
        self.layer4 = Dense(hidden_size, activation=LeakyReLU(alpha=0.3), dtype=tf.float32)
        self.layer5 = Dense(hidden_size, activation=LeakyReLU(alpha=0.3), dtype=tf.float32)
        self.layer6 = Dense(hidden_size, activation=LeakyReLU(alpha=0.3), dtype=tf.float32)

        self.output_layer = Dense(self.num_outputs, activation="linear", dtype=tf.float32)

    def call(self, msi):
        H = tf.shape(msi)[0]
        W = tf.shape(msi)[1]

        x = tf.reshape(msi, [H * W, -1])

        x = self.layer1(x)
        x1 = x
        x = self.layer2(x) + x1

        x2 = x
        x = self.layer3(x)
        x = self.layer4(x) + x2

        x3 = x
        x = self.layer5(x)
        x = self.layer6(x) + x3

        hsi = self.output_layer(x)
        hsi = tf.reshape(hsi, [H, W, self.num_outputs])
        return hsi


def train_spectral_mlp(
    lr_msi,
    lr_hsi,
    epochs=2500,
    lr_schedule="one_cycle",
    init_lr=1e-4,
    max_lr=1e-2,
    final_lr=1e-6,
    min_lr=1e-6,
    num_restarts=1,
    hidden_size=64,
    batch_size=1024,
):
    """Train the SpectraLift spectral MLP."""
    h, w, C = lr_hsi.shape
    model = SpectralSR_MLP(num_outputs=C, hidden_size=hidden_size)

    def get_lr(epoch):
        if lr_schedule == "one_cycle":
            pct_up = 0.3
            if epoch < pct_up * epochs:
                return init_lr + (max_lr - init_lr) * (epoch / (pct_up * epochs))
            return max_lr - (max_lr - final_lr) * (
                (epoch - pct_up * epochs) / ((1 - pct_up) * epochs)
            )
        if lr_schedule == "cosine_restart":
            period = epochs // num_restarts
            cur = epoch % period
            cos_decay = 0.5 * (1 + math.cos(math.pi * cur / period))
            return min_lr + (max_lr - min_lr) * cos_decay
        return init_lr

    optimizer = tf.keras.optimizers.Adam(learning_rate=init_lr)
    loss_fn = tf.keras.losses.MeanAbsoluteError()

    pbar = tqdm(range(1, epochs + 1), desc="Training SR-MLP", unit="epoch")
    for epoch in pbar:
        lr = get_lr(epoch)
        optimizer.learning_rate.assign(lr)

        if batch_size is None:
            with tf.GradientTape() as tape:
                pred = model(lr_msi)
                loss = loss_fn(lr_hsi, pred)
            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))
            pbar.set_postfix(loss=f"{loss.numpy():.4f}", lr=f"{lr:.2e}")
        else:
            epoch_loss = 0.0
            count = 0
            for y0 in range(0, h, batch_size):
                for x0 in range(0, w, batch_size):
                    sub_msi = lr_msi[y0:y0 + batch_size, x0:x0 + batch_size, :]
                    sub_hsi = lr_hsi[y0:y0 + batch_size, x0:x0 + batch_size, :]
                    with tf.GradientTape() as tape:
                        pred = model(sub_msi)
                        loss = loss_fn(sub_hsi, pred)
                    grads = tape.gradient(loss, model.trainable_variables)
                    optimizer.apply_gradients(zip(grads, model.trainable_variables))

                    epoch_loss += loss.numpy()
                    count += 1

            avg_loss = epoch_loss / count if count else 0.0
            pbar.set_postfix(average_loss=f"{avg_loss:.4f}", lr=f"{lr:.2e}")

    return model


def run_pipeline(
    HR_MSI,
    LR_HSI,
    srf,
    psf=None,
    metadata=None,
    lr_schedule="one_cycle",
    initial_lr=1e-3,
    max_lr=1e-2,
    final_lr=1e-6,
    min_lr=1e-6,
    num_restarts=2,
    sin_hidden_size=64,
    num_epochs=2500,
    training_batch_size=None,
    inference_batch_size=None,
):
    """
    HyperBench-facing SpectraLift pipeline.

    HyperBench computes reconstruction metrics itself after receiving the
    prediction. This function therefore returns:
    - the prediction
    - model statistics
    """
    training_start = time.perf_counter()

    # SpectraLift performs its own NumPy -> TensorFlow conversion internally.
    hr_msi, lr_hsi, lr_msi, srf_tf = prepare_inputs(HR_MSI, LR_HSI, srf)

    trained_spectral_sr_mlp = train_spectral_mlp(
        lr_msi,
        lr_hsi,
        epochs=num_epochs,
        lr_schedule=lr_schedule,
        init_lr=initial_lr,
        max_lr=max_lr,
        final_lr=final_lr,
        min_lr=min_lr,
        num_restarts=num_restarts,
        hidden_size=sin_hidden_size,
        batch_size=training_batch_size,
    )

    training_end = time.perf_counter()
    training_time = training_end - training_start

    if inference_batch_size is None:
        SR_image_tf, num_params, flops, mem_used, inference_time = (
            infer_and_analyze_model_performance_tf(
                trained_spectral_sr_mlp,
                sample_inputs=[hr_msi],
            )
        )
        SR_image = tf_to_numpy(SR_image_tf)
    else:
        inference_start = time.perf_counter()
        SR_image = batched_inference(
            trained_spectral_sr_mlp,
            HR_MSI,
            inference_batch_size,
        )
        inference_end = time.perf_counter()

        num_params = int(np.sum([np.prod(v.shape) for v in trained_spectral_sr_mlp.trainable_variables]))
        flops = float("nan")
        mem_used = float("nan")
        inference_time = inference_end - inference_start

    # SpectraLift already handles output clipping internally here, which shows
    # that HyperBench can work with models that manage this on their own.
    SR_image = np.clip(SR_image, 0.0, 1.0)

    stats = {
        "framework": "tensorflow",
        "training_time_sec": float(training_time),
        "inference_time_sec": float(inference_time),
        "num_parameters": int(num_params),
        "flops": float(flops) if not isinstance(flops, (int, np.integer)) else int(flops),
        "gpu_memory_mb": float(mem_used),
    }

    return SR_image, stats

class SpectraLiftPipeline:
    def run_pipeline(self, HR_MSI, LR_HSI, srf, psf=None, metadata=None):
        return run_pipeline(HR_MSI, LR_HSI, srf, psf, metadata)

## Wrap the method with `PipelineAdapter`

`PipelineAdapter` is the standard integration path for the `run_pipeline(...)` interface.

In [4]:
adapter = PipelineAdapter(
        pipeline=SpectraLiftPipeline(),
        name="SpectraLift",
        input_backend="numpy",
        output_backend="numpy",
        add_batch_dim=False,
        device="auto",
    )

print("Adapter ready:", adapter.name)

Adapter ready: SpectraLift


## Part 1: Explicit benchmark design with `degradation_specs`

This approach is useful when the experiment set is known in advance and should be controlled exactly.

DegradationSpec allows a user to control the exact downsampling_ratio, msi_band_count, and the Signal to Noise ratios for noise injection into the lr_hsi and hr_msi. A user can specify the exact PSFs that they wish to run these degradations with when they build a benchmark configuration (shown below).

In [5]:
explicit_specs = [
    DegradationSpec(downsample_ratio=4,  msi_band_count=4,  spatial_snr_db=35.0, spectral_snr_db=40.0),
    DegradationSpec(downsample_ratio=8,  msi_band_count=4,  spatial_snr_db=30.0, spectral_snr_db=40.0),
    DegradationSpec(downsample_ratio=16, msi_band_count=4,  spatial_snr_db=25.0, spectral_snr_db=40.0),
    DegradationSpec(downsample_ratio=32, msi_band_count=4,  spatial_snr_db=20.0, spectral_snr_db=40.0),
    DegradationSpec(downsample_ratio=8,  msi_band_count=3,  spatial_snr_db=30.0, spectral_snr_db=40.0),
    DegradationSpec(downsample_ratio=8,  msi_band_count=8,  spatial_snr_db=30.0, spectral_snr_db=40.0),
    DegradationSpec(downsample_ratio=8,  msi_band_count=16, spatial_snr_db=30.0, spectral_snr_db=40.0),
]

for spec in explicit_specs:
    print(spec)

DegradationSpec(downsample_ratio=4, msi_band_count=4, spatial_snr_db=35.0, spectral_snr_db=40.0)
DegradationSpec(downsample_ratio=8, msi_band_count=4, spatial_snr_db=30.0, spectral_snr_db=40.0)
DegradationSpec(downsample_ratio=16, msi_band_count=4, spatial_snr_db=25.0, spectral_snr_db=40.0)
DegradationSpec(downsample_ratio=32, msi_band_count=4, spatial_snr_db=20.0, spectral_snr_db=40.0)
DegradationSpec(downsample_ratio=8, msi_band_count=3, spatial_snr_db=30.0, spectral_snr_db=40.0)
DegradationSpec(downsample_ratio=8, msi_band_count=8, spatial_snr_db=30.0, spectral_snr_db=40.0)
DegradationSpec(downsample_ratio=8, msi_band_count=16, spatial_snr_db=30.0, spectral_snr_db=40.0)


## Build an explicit `BenchmarkConfig`

This configuration uses:

- ten PSFs
- one sigma
- one kernel radius
- exact `degradation_specs`
- clipping disabled
- CSV writing enabled
- per-case flushing enabled

In [6]:
explicit_csv_path = OUTPUT_DIR / "SpectraLift_jupyter_exact_degradations.csv"

explicit_config = BenchmarkConfig(
    scene_path=SCENE_PATH,
    scene_key=SCENE_KEY,
    psf_names=["gaussian"], # Users can add any of the supported PSFs here
    psf_sigmas=[3.4],
    psf_kernel_radii=[7],
    degradation_specs=explicit_specs,
    clip_prediction_to_unit_interval=False,
    output_csv_path=explicit_csv_path,
    save_csv=True,
    flush_csv_after_each_case=True,
    overwrite_csv_on_start=True,
    fail_fast=True,
)

## Run the explicit benchmark

In [7]:
explicit_results = run_benchmark(adapter, explicit_config)

print("Number of explicit result rows:", len(explicit_results.rows))
print("CSV written to:", explicit_csv_path.resolve())

[2026-03-31 11:09:16] INFO | hyperbench | Loading scene from '../../data/DC_data.mat' with key 'dc'
[2026-03-31 11:09:17] INFO | hyperbench | Generating benchmark cases from config.
[2026-03-31 11:09:17] INFO | hyperbench | Initialized benchmark CSV at 'outputs/SpectraLift_jupyter_exact_degradations.csv'.
[2026-03-31 11:09:17] INFO | hyperbench | Starting benchmark run with 7 case(s) using adapter 'SpectraLift'.
[2026-03-31 11:09:17] INFO | hyperbench | Starting case_00000 | psf=gaussian, sigma=3.4, kernel=7, r=4, c=4, snr_spatial=35.0, snr_spectral=40.0, seed=42
2026-03-31 11:10:01.268442: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9611 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:3e:00.0, compute capability: 7.5
/home/rgshah_umass_edu/miniconda3/envs/hyperbench/lib/python3.9/site-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is depreca

Instructions for updating:
This API was designed for TensorFlow v1. See https://www.tensorflow.org/guide/migrate for instructions on how to migrate your code to TensorFlow v2.


[2026-03-31 11:14:57] INFO | hyperbench | Completed case_00000 successfully in 339.877s | RMSE=0.011530, PSNR=38.676177, SSIM=0.991123, UIQI=0.978000, ERGAS=3.549713, SAM=1.788504
[2026-03-31 11:14:57] INFO | hyperbench | Starting case_00001 | psf=gaussian, sigma=3.4, kernel=7, r=8, c=4, snr_spatial=30.0, snr_spectral=40.0, seed=42
Training SR-MLP: 100%|██████████| 2500/2500 [02:33<00:00, 16.34epoch/s, loss=0.0068, lr=1.00e-06]
I0000 00:00:1774955893.923217 1785315 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
2026-03-31 11:18:13.923364: I tensorflow/core/grappler/clusters/single_machine.cc:361] Starting new session
2026-03-31 11:18:13.928593: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9611 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:3e:00.0, compute capability: 7.5
[2026-03-31 11:20:19] INFO | hyperbench | Completed case_00001 success

Number of explicit result rows: 7
CSV written to: /work/pi_mduarte_umass_edu/Ritik_Shah_Folder/HyperBench/examples/SpectraLift/outputs/SpectraLift_jupyter_exact_degradations.csv


## Inspect the structured benchmark rows

Each row is a dictionary containing:

- scene and method identifiers
- degradation parameters
- input shapes
- metrics
- status information
- runtime
- optional stats returned by the pipeline

In [8]:
for i, row in enumerate(explicit_results.rows):
    print(f"Row {i}")
    for key, value in row.items():
        print(f"  {key}: {value}")
    print("-" * 100)

Row 0
  scene_name: DC_data
  shape_policy: strict
  method_name: SpectraLift
  psf_name: gaussian
  sigma: 3.4
  kernel_size: 15
  downsampling_ratio: 4
  msi_band_count: 4
  spatial_snr: 35.0
  spectral_snr: 40.0
  fwhm_factor: 4.2
  seed: 42
  gt_shape: (1280, 307, 191)
  lr_hsi_shape: (320, 76, 191)
  hr_msi_shape: (1280, 307, 4)
  prediction_clipped: False
  prediction_clip_min: 0.0
  prediction_clip_max: 1.0
  framework: tensorflow
  training_time_sec: 169.20118175633252
  inference_time_sec: 0.00932374782860279
  num_parameters: 33535
  flops: 26205323521
  gpu_memory_mb: 286.3134765625
  RMSE: 0.011530168599823956
  PSNR: 38.676177078719476
  SSIM: 0.9911225868466446
  UIQI: 0.9779997364442636
  ERGAS: 3.5497134352039077
  SAM: 1.7885040044784546
  status: success
  error: 
  runtime_seconds: 339.8765777037479
----------------------------------------------------------------------------------------------------
Row 1
  scene_name: DC_data
  shape_policy: strict
  method_name: Spe

## Part 2: Sweep-style benchmark design

A sweep-style configuration is useful when you want HyperBench to build the experiment set automatically from parameter lists.

This style is appropriate when the benchmark should cover a broader grid of conditions. This example keeps the sweep deliberately simple since sweep style benchmarking generates too many experiments.

In [5]:
sweep_csv_path = OUTPUT_DIR / "SpectraLift_jupyter_sweep_results.csv"

sweep_config = BenchmarkConfig(
    scene_path=SCENE_PATH,
    scene_key=SCENE_KEY,
    psf_names=["gaussian", "moffat"],
    psf_sigmas=[3.4],
    psf_kernel_radii=[7],
    msi_band_counts=[3, 4, 8],
    downsample_ratio_to_spatial_snr_db={
        4: 35.0,
        8: 30.0,
    },
    spectral_snr_dbs=[40.0],
    clip_prediction_to_unit_interval=True,
    prediction_clip_min=0.0,
    prediction_clip_max=1.0,
    output_csv_path=sweep_csv_path,
    save_csv=True,
    flush_csv_after_each_case=True,
    overwrite_csv_on_start=True,
    fail_fast=True,
)

print(sweep_config)

BenchmarkConfig(scene_path=PosixPath('../../data/DC_data.mat'), scene_key='dc', psf_names=['gaussian', 'moffat'], psf_sigmas=[3.4], psf_kernel_radii=[7], degradation_specs=None, msi_band_counts=[3, 4, 8], downsample_ratio_to_spatial_snr_db={4: 35.0, 8: 30.0}, spectral_snr_dbs=[40.0], fwhm_factor=4.2, seed=42, metrics=['rmse', 'psnr', 'ssim', 'uiqi', 'ergas', 'sam'], normalize_inputs=True, lower_percentile=1.0, upper_percentile=99.0, clip_prediction_to_unit_interval=True, prediction_clip_min=0.0, prediction_clip_max=1.0, save_csv=True, output_csv_path=PosixPath('outputs/SpectraLift_jupyter_sweep_results.csv'), flush_csv_after_each_case=True, overwrite_csv_on_start=True, fail_fast=True, user_srf=None, user_psf=None, metadata={})


## Run the sweep benchmark

In [6]:
sweep_results = run_benchmark(adapter, sweep_config)

print("Number of sweep result rows:", len(sweep_results.rows))
print("CSV written to:", sweep_csv_path.resolve())

[2026-03-31 12:10:47] INFO | hyperbench | Loading scene from '../../data/DC_data.mat' with key 'dc'
[2026-03-31 12:10:48] INFO | hyperbench | Generating benchmark cases from config.
[2026-03-31 12:10:48] INFO | hyperbench | Initialized benchmark CSV at 'outputs/SpectraLift_jupyter_sweep_results.csv'.
[2026-03-31 12:10:48] INFO | hyperbench | Starting benchmark run with 12 case(s) using adapter 'SpectraLift'.
[2026-03-31 12:10:48] INFO | hyperbench | Starting case_00000 | psf=gaussian, sigma=3.4, kernel=7, r=4, c=3, snr_spatial=35.0, snr_spectral=40.0, seed=42
2026-03-31 12:11:36.164292: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9611 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:b1:00.0, compute capability: 7.5
/home/rgshah_umass_edu/miniconda3/envs/hyperbench/lib/python3.9/site-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated.

Instructions for updating:
This API was designed for TensorFlow v1. See https://www.tensorflow.org/guide/migrate for instructions on how to migrate your code to TensorFlow v2.


[2026-03-31 12:16:35] INFO | hyperbench | Completed case_00000 successfully in 346.815s | RMSE=0.015429, PSNR=36.177249, SSIM=0.987724, UIQI=0.979486, ERGAS=3.542411, SAM=2.119377
[2026-03-31 12:16:35] INFO | hyperbench | Starting case_00001 | psf=gaussian, sigma=3.4, kernel=7, r=4, c=4, snr_spatial=35.0, snr_spectral=40.0, seed=42
Training SR-MLP: 100%|██████████| 2500/2500 [02:31<00:00, 16.52epoch/s, loss=0.0047, lr=1.00e-06]
I0000 00:00:1774959594.627045  548630 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
2026-03-31 12:19:54.627205: I tensorflow/core/grappler/clusters/single_machine.cc:361] Starting new session
2026-03-31 12:19:54.632487: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9611 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:b1:00.0, compute capability: 7.5
[2026-03-31 12:22:00] INFO | hyperbench | Completed case_00001 success

Number of sweep result rows: 12
CSV written to: /work/pi_mduarte_umass_edu/Ritik_Shah_Folder/HyperBench/examples/SpectraLift/outputs/SpectraLift_jupyter_sweep_results.csv


## Inspect the sweep rows

In [7]:
for i, row in enumerate(sweep_results.rows):
    print(f"Sweep row {i}")
    for key, value in row.items():
        print(f"  {key}: {value}")
    print("-" * 100)

Sweep row 0
  scene_name: DC_data
  shape_policy: strict
  method_name: SpectraLift
  psf_name: gaussian
  sigma: 3.4
  kernel_size: 15
  downsampling_ratio: 4
  msi_band_count: 3
  spatial_snr: 35.0
  spectral_snr: 40.0
  fwhm_factor: 4.2
  seed: 42
  gt_shape: (1280, 307, 191)
  lr_hsi_shape: (320, 76, 191)
  hr_msi_shape: (1280, 307, 3)
  prediction_clipped: True
  prediction_clip_min: 0.0
  prediction_clip_max: 1.0
  framework: tensorflow
  training_time_sec: 165.0310331779765
  inference_time_sec: 0.009836392942816019
  num_parameters: 33471
  flops: 26155024641
  gpu_memory_mb: 286.3134765625
  RMSE: 0.01542900345986978
  PSNR: 36.177249411041785
  SSIM: 0.9877236587591414
  UIQI: 0.9794856864738162
  ERGAS: 3.5424114404533156
  SAM: 2.1193771362304688
  status: success
  error: 
  runtime_seconds: 346.81520303792786
----------------------------------------------------------------------------------------------------
Sweep row 1
  scene_name: DC_data
  shape_policy: strict
  metho